# Задание 1. Оптимальное размещение виртуальных машин

**Курс:** Оптимизационные модели. Моделирование и решение.

Задача управления ресурсами ЦОД: разместить VM на PM с минимальными затратами.

## Математическая модель

**Переменные:** $x_{ij}, y_j \in \{0,1\}$

**Цель:** $\min \alpha \sum_j c_j y_j + \beta \sum_{i,j} c_{ij} x_{ij}$ (в эталонном MPS для $y_j$ используется коэффициент $\beta c_j$; при сверке с MPS см. последнюю ячейку)

**Ограничения:**
1. $\sum_j x_{ij} = 1$ — каждая VM на одном сервере
2. $\sum_i d_{ir} x_{ij} \le q_{jr} y_j$ — ресурсы
3. $x_{i_1 j} + x_{i_2 j} \le 1$ для пар $(4,5), (6,7), (8,9)$
4. $\sum_{j \in J_g} c_j y_j + \sum_{i,j \in J_g} c_{ij} x_{ij} = Plan_g$
5. $\sum_{(i,j) \in \mathcal{L}} x_{ij} \ge \varepsilon$ — миграция

In [ ]:
# using Pkg; Pkg.add(["JuMP", "HiGHS", "Plots", "Statistics"])

using JuMP
using HiGHS
using Statistics
using Plots

const DATA_DIR = let
    for base in (try @__DIR__ catch; pwd() end, pwd())
        candidate = joinpath(base, "data")
        isfile(joinpath(candidate, "general_info.txt")) && return abspath(candidate)
    end
    error("Папка data не найдена. Откройте блокнот из каталога lab_1.")
end

In [ ]:
function parse_general_info(path)
    text = read(path, String)
    params = Dict{String, Any}()
    for m in eachmatch(r"([^=,]+)\s*=\s*([\d.]+)", text)
        params[strip(m.captures[1])] = parse(Float64, m.captures[2])
    end
    params["n"] = Int(params["n"])
    params["m"] = Int(params["m"])
    params["G"] = Int(params["|G|"])
    params["locVMs"] = Int(params["locVMs"])
    params["R"] = Int(params["|R|"])
    params["V"] = Int(params["|V|"])
    params["eps"] = Int(params["eps"])

    plan_vals = Float64[]
    in_plan = false
    for line in split(text, '\n')
        s = strip(line)
        if occursin("Plan_Gg", s)
            in_plan = true
            continue
        end
        in_plan || continue
        isempty(s) && continue
        s == "]" && break
        try
            push!(plan_vals, parse(Float64, replace(s, "[" => "", "]" => "")))
        catch
            break
        end
    end
    params["Plan_G"] = plan_vals[1:Int(params["G"])]
    params["conflict_pairs"] = [(4, 5), (6, 7), (8, 9)]
    return params
end

function read_matrix(path)
    rows = Vector{Vector{Float64}}()
    for line in eachline(path)
        s = strip(line)
        isempty(s) && continue
        push!(rows, parse.(Float64, split(s)))
    end
    return hcat(rows...)'
end

function read_J_for_G(path)
    text = read(path, String)
    i = findfirst("[[", text)
    j = findlast("]]", text)
    inner = text[i[2]+1:j[1]-1]
    servers = Vector{Int}[]
    for chunk in split(inner, "], [")
        chunk = strip(replace(chunk, "[" => "", "]" => ""))
        isempty(chunk) && continue
        push!(servers, parse.(Int, split(chunk, ",")))
    end
    return servers
end

function read_initial_locations(path)
    locs = Tuple{Int, Int}[]
    for line in eachline(path)
        s = strip(line)
        occursin(r"^\(\d+\s*,\s*\d+\)", s) || continue
        m = match(r"\((\d+)\s*,\s*(\d+)\)", s)
        push!(locs, (parse(Int, m.captures[1]), parse(Int, m.captures[2])))
    end
    return locs
end

function load_data(data_dir=DATA_DIR)
    info = parse_general_info(joinpath(data_dir, "general_info.txt"))
    c_j = read_matrix(joinpath(data_dir, "c_y_j.txt"))[:]
    c_ij = read_matrix(joinpath(data_dir, "C_x_ij.txt"))
    d = read_matrix(joinpath(data_dir, "D_ir.txt"))
    q = read_matrix(joinpath(data_dir, "Q_jr.txt"))
    J_for_G = read_J_for_G(joinpath(data_dir, "J_for_G.txt"))
    initial_locs = read_initial_locations(joinpath(data_dir, "x_ij_firstlyLocated.txt"))

    n, m = info["n"], info["m"]
    @assert size(c_ij) == (n, m)
    @assert size(d) == (n, info["R"])
    @assert size(q) == (m, info["R"])
    @assert length(c_j) == m
    @assert length(J_for_G) == info["G"]

    return (
        n = n,
        m = m,
        G = info["G"],
        R = info["R"],
        alpha = info["alpha"],
        beta = info["beta"],
        eps = info["eps"],
        Plan_G = info["Plan_G"],
        conflict_pairs = info["conflict_pairs"],
        c_j = c_j,
        c_ij = c_ij,
        d = d,
        q = q,
        J_for_G = J_for_G,
        initial_locs = initial_locs,
    )
end

data = load_data()
println("Загружено: n=$(data.n), m=$(data.m), |G|=$(data.G), исходных размещений=$(length(data.initial_locs))")

In [ ]:
function build_vm_model(data; time_limit=3600.0)
    n, m = data.n, data.m
    alpha, beta, eps = data.alpha, data.beta, data.eps

    model = Model(HiGHS.Optimizer)
    set_attribute(model, "time_limit", time_limit)
    set_silent(model)

    allowed = [(i, j) for i in 1:n, j in 1:m if data.c_ij[i, j] > 0]
    allowed_j = [Int[] for _ in 1:n]
    allowed_i = [Int[] for _ in 1:m]
    for (i, j) in allowed
        push!(allowed_j[i], j)
        push!(allowed_i[j], i)
    end

    @variable(model, y[1:m], Bin)
    @variable(model, x[i=1:n, j=1:m], Bin)

    for i in 1:n, j in 1:m
        if data.c_ij[i, j] == 0
            @constraint(model, x[i, j] == 0)
        end
    end

    @objective(
        model,
        Min,
        alpha * sum(data.c_j[j] * y[j] for j in 1:m) +
            beta * sum(data.c_ij[i, j] * x[i, j] for (i, j) in allowed),
    )

    for i in 1:n
        @constraint(model, sum(x[i, j] for j in allowed_j[i]) == 1)
    end

    for j in 1:m, r in 1:data.R
        @constraint(
            model,
            sum(data.d[i, r] * x[i, j] for i in allowed_i[j]) <= data.q[j, r] * y[j],
        )
    end

    for (i1, i2) in data.conflict_pairs, j in 1:m
        if data.c_ij[i1, j] > 0 && data.c_ij[i2, j] > 0
            @constraint(model, x[i1, j] + x[i2, j] <= 1)
        end
    end

    for (g, servers) in enumerate(data.J_for_G)
        @constraint(
            model,
            sum(data.c_j[j] * y[j] for j in servers) +
            sum(data.c_ij[i, j] * x[i, j] for i in 1:n, j in servers) == data.Plan_G[g],
        )
    end

    @constraint(
        model,
        sum(x[i, j] for (i, j) in data.initial_locs if data.c_ij[i, j] > 0) >= eps,
    )

    return model
end

model = build_vm_model(data)
println("Модель построена. Переменных: ", num_variables(model))

In [ ]:
println("Запуск HiGHS...")
optimize!(model)

term = termination_status(model)
println("Статус: ", term)
if has_values(model)
    println("Z = ", objective_value(model))
else
    @warn "Решение не получено — увеличьте time_limit или используйте Gurobi/CPLEX."
end

In [ ]:
function extract_solution(model, data)
    n, m = data.n, data.m
    placement = Dict{Int, Int}()
    for i in 1:n, j in 1:m
        if data.c_ij[i, j] > 0 && value(model[:x][i, j]) > 0.5
            placement[i] = j
        end
    end
    active_servers = [j for j in 1:m if value(model[:y][j]) > 0.5]
    stayed = count(((i, j),) -> get(placement, i, -1) == j, data.initial_locs)
    return placement, active_servers, stayed
end

if has_values(model)
    placement, active_servers, stayed = extract_solution(model, data)
    println("Активных серверов: ", length(active_servers), " / ", data.m)
    println("VM на исходных местах: ", stayed, " (>= ", data.eps, ")")
end

In [ ]:
if has_values(model)
    placement, active_servers, stayed = extract_solution(model, data)

    load_per_server = zeros(data.m)
    for (i, j) in placement
        load_per_server[j] += 1
    end

    bar(1:data.m, load_per_server; xlabel="Сервер", ylabel="Число VM", title="VM на серверах", legend=false)
    migrated = data.n - stayed
    pie(["На месте ($stayed)", "Мигрировали ($migrated)"], title="Миграция")
end

## Проверка через MPS-файл

In [ ]:
function solve_mps(path; time_limit=3600.0)
    mps_model = read_from_file(path)
    set_optimizer(mps_model, HiGHS.Optimizer)
    set_attribute(mps_model, "time_limit", time_limit)
    set_silent(mps_model)
    optimize!(mps_model)
    println("MPS статус: ", termination_status(mps_model))
    has_values(mps_model) && println("MPS objective = ", objective_value(mps_model))
    return mps_model
end

# Основной рабочий способ (рекомендуется):
mps_model = solve_mps(joinpath(DATA_DIR, "model.mps"))